In [16]:
import pandas as pd
import numpy as np

# ── helpers ──────────────────────────────────────────────────────────────────

def summary_stats(s: pd.Series) -> dict:
    """Mean, median, sd, p10, p90 for a numeric series (NaN-safe)."""
    s = s.dropna()
    if s.empty:
        return dict(mean=np.nan, median=np.nan, sd=np.nan, p10=np.nan, p90=np.nan)
    return dict(
        mean=s.mean(),
        median=s.median(),
        sd=s.std(ddof=1),
        p10=s.quantile(0.10),
        p90=s.quantile(0.90),
    )

def trend_slope(s: pd.Series) -> float:
    """OLS slope (value per day) over a time-indexed series, using actual date gaps."""
    s = s.dropna()
    if len(s) < 2:
        return np.nan
    # BUG FIX (Bug 7): use actual date ordinals if the index is DatetimeIndex,
    # otherwise fall back to positional integers
    if isinstance(s.index, pd.DatetimeIndex):
        x = (s.index - s.index[0]).days.values.astype(float)
    else:
        x = np.arange(len(s), dtype=float)
    return float(np.polyfit(x, s.values, 1)[0])

def mad(s: pd.Series) -> float:
    """Mean absolute day-over-day change."""
    s = s.dropna()
    diffs = s.diff().abs().dropna()
    return diffs.mean() if not diffs.empty else np.nan

def stats_block(series: pd.Series, prefix: str) -> dict:
    """All 7 summary metrics for one series, with a prefix."""
    out = {f"{prefix}_{k}": v for k, v in summary_stats(series).items()}
    out[f"{prefix}_slope"] = trend_slope(series)
    out[f"{prefix}_mad"]   = mad(series)
    return out

In [28]:
activity   = pd.read_csv("/home/jupyter/workspaces/result/final/classifier/final_all_activity.txt",   sep="\t", parse_dates=["date"])
sleep      = pd.read_csv("/home/jupyter/workspaces/result/final/classifier/final_all_sleep.txt",      sep="\t", parse_dates=["date"])
hr_summary = pd.read_csv("/home/jupyter/workspaces/result/final/classifier/final_all_hr.txt", sep="\t", parse_dates=["date"])

print(len(activity['person_id'].unique()))
print(len(sleep['person_id'].unique()))
print(len(hr_summary['person_id'].unique()))

activity  = activity.sort_values(["person_id", "date"])
sleep      = sleep.sort_values(["person_id", "date"])
hr_summary = hr_summary.sort_values(["person_id", "date"])

activity = activity.drop_duplicates()
sleep = sleep.drop_duplicates()
hr_summary = hr_summary.drop_duplicates()


all_patients = sorted(
    set(activity["person_id"]) | set(sleep["person_id"]) | set(hr_summary["person_id"])
)

# ── feature extraction ────────────────────────────────────────────────────────

ACTIVITY_COLS = [
    "activity_calories", "calories_bmr", "calories_out", "elevation",
    "fairly_active_minutes", "lightly_active_minutes", "marginal_calories",
    "sedentary_minutes", "very_active_minutes",
]

SLEEP_COLS = [
    "minute_in_bed", "minute_deep", "minute_light",
    "minute_rem", "minute_wake",
]

HR_ZONES = ["Out of Range", "Fat Burn", "Cardio", "Peak"]

8926
8926
8926


In [19]:
def extract_patient(pid):
    row = {"person_id": pid}

    # ── 1. Activity features ────────────────────────────────────────────────
    act = activity[activity["person_id"] == pid].sort_values("date")
    
    print(f"[{pid}] activity: {len(act)} rows")
    
    act_dated = act.set_index("date")
    for col in ACTIVITY_COLS:
        if col in act_dated.columns:
            row.update(stats_block(act_dated[col], col))

    # Steps extras
    if "steps" in act_dated.columns:
        row.update(stats_block(act_dated["steps"], "steps"))

        valid_steps = act_dated["steps"].dropna()
        n_valid = len(valid_steps)
        row["steps_pct_days_lt_5000"] = (
            (valid_steps < 5000).sum() / n_valid * 100 if n_valid > 0 else np.nan
        )
    print(f"[{pid}] activity features done")
    
    # ── 2. Sleep features ───────────────────────────────────────────────────
    main_sleep = sleep[(sleep["person_id"] == pid) & (sleep["is_main_sleep"] == True)].copy()
    nap_sleep  = sleep[(sleep["person_id"] == pid) & (sleep["is_main_sleep"] == False)].copy()
    print(f"[{pid}] sleep: {len(main_sleep)} main, {len(nap_sleep)} nap records")
 
    all_sleep = sleep[sleep["person_id"] == pid].copy()
    daily_total = all_sleep.groupby("date")["minute_asleep"].sum(min_count=1)

    row.update(stats_block(daily_total, "daily_total_sleep"))
 

    if not main_sleep.empty:
        main_sleep_dated = main_sleep.set_index("date")
        for col in SLEEP_COLS:
            if col in main_sleep_dated.columns:
                row.update(stats_block(main_sleep_dated[col], col))
    else:
        print(f"[{pid}] WARNING: no main sleep records, sleep features set to NaN")
        for col in SLEEP_COLS:
            for sfx in ["mean", "median", "sd", "p10", "p90", "slope", "mad"]:
                row[f"{col}_{sfx}"] = np.nan
 
    # Sleep efficiency: collapse nap+main per day, minute_asleep / minute_in_bed * 100
    if "minute_asleep" in all_sleep.columns and "minute_in_bed" in all_sleep.columns:
        daily_eff = all_sleep.groupby("date")[["minute_asleep", "minute_in_bed"]].sum(min_count=1)
        eff = np.where(
            daily_eff["minute_in_bed"] > 0,
            daily_eff["minute_asleep"] / daily_eff["minute_in_bed"] * 100,
            np.nan,
        )
        row.update(stats_block(pd.Series(eff, index=daily_eff.index, name="sleep_efficiency"), "sleep_efficiency"))
 
    # Nap indicators
    nap_days = nap_sleep["date"].nunique()
    row["nap_indicator"]   = int(nap_days > 0)
    # denominator: union of all days with any record for this patient
    act_days  = set(act_dated.index.normalize()) if not act_dated.empty else set()
    slp_days  = set(sleep[sleep["person_id"] == pid]["date"].dt.normalize())
    total_obs_days = len(act_days | slp_days)
    row["nap_pct_days"]    = (nap_days / total_obs_days * 100) if total_obs_days > 0 else np.nan
    if not nap_sleep.empty and "minute_in_bed" in nap_sleep.columns:
        row["nap_mean_duration_min"] = nap_sleep["minute_in_bed"].mean()
    else:
        row["nap_mean_duration_min"] = np.nan
    print(f"[{pid}] sleep features done")
 
    
    # ── 3. Heart-rate zone features ─────────────────────────────────────────
    hr_pid = hr_summary[hr_summary["person_id"] == pid]
    
    print(f"[{pid}] HR: {len(hr_pid)} rows across {hr_pid['zone_name'].nunique()} zones")

    for zone in HR_ZONES:
        zone_df = hr_pid[hr_pid["zone_name"] == zone].sort_values("date")
        prefix  = "hr_" + zone.lower().replace(" ", "_")
        zone_dated = zone_df.set_index("date")
        for metric in ["min_heart_rate", "max_heart_rate"]:
            if metric in zone_dated.columns:
                row.update(stats_block(zone_dated[metric], f"{prefix}_{metric}"))
            else:
                for sfx in ["mean","median","sd","p10","p90","slope","mad"]:
                    row[f"{prefix}_{metric}_{sfx}"] = np.nan
    print(f"[{pid}] HR features done")

    return row

In [20]:
# ── run and save ──────────────────────────────────────────────────────────────

records = [extract_patient(pid) for pid in all_patients]
features = pd.DataFrame(records)

#output_path = "fitbit_features.xlsx"
#features.to_excel(output_path, index=False)

print(f"Done. Shape: {features.shape}")
print("\nColumn preview:")


[1000464] activity: 102 rows
[1000464] activity features done
[1000464] sleep: 85 main, 15 nap records
[1000464] sleep features done
[1000464] HR: 352 rows across 4 zones
[1000464] HR features done
[1000883] activity: 107 rows
[1000883] activity features done
[1000883] sleep: 103 main, 30 nap records
[1000883] sleep features done
[1000883] HR: 416 rows across 4 zones
[1000883] HR features done
[1001942] activity: 70 rows
[1001942] activity features done
[1001942] sleep: 68 main, 10 nap records
[1001942] sleep features done
[1001942] HR: 276 rows across 4 zones
[1001942] HR features done
[1003126] activity: 2791 rows
[1003126] activity features done
[1003126] sleep: 2680 main, 90 nap records
[1003126] sleep features done
[1003126] HR: 5564 rows across 4 zones
[1003126] HR features done
[1003304] activity: 108 rows
[1003304] activity features done
[1003304] sleep: 92 main, 17 nap records
[1003304] sleep features done
[1003304] HR: 420 rows across 4 zones
[1003304] HR features done
[10037

NameError: name 'output_path' is not defined

In [21]:
features.to_csv("/home/jupyter/workspaces/result/final/classifier/fitbit_features.txt", sep="\t", index=False)

In [26]:
len(activity['person_id'].unique())

8926

In [30]:
print(features.columns.tolist()[:30])

['person_id', 'activity_calories_mean', 'activity_calories_median', 'activity_calories_sd', 'activity_calories_p10', 'activity_calories_p90', 'activity_calories_slope', 'activity_calories_mad', 'calories_bmr_mean', 'calories_bmr_median', 'calories_bmr_sd', 'calories_bmr_p10', 'calories_bmr_p90', 'calories_bmr_slope', 'calories_bmr_mad', 'calories_out_mean', 'calories_out_median', 'calories_out_sd', 'calories_out_p10', 'calories_out_p90', 'calories_out_slope', 'calories_out_mad', 'elevation_mean', 'elevation_median', 'elevation_sd', 'elevation_p10', 'elevation_p90', 'elevation_slope', 'elevation_mad', 'fairly_active_minutes_mean']
